# Nonlinear Driving Terms Along the Ring

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next(
    p for p in (cwd, *cwd.parents)
    if (p / "simplestoragering").exists() and (p / "examples").exists()
)

sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "examples"))

print(f"Project root: {project_root}")

In [ ]:
import simplestoragering as ssr
import numpy as np
import matplotlib.pyplot as plt
import time
from example_lattice import generate_ring


ring = generate_ring()
ring.linear_optics()

## Compute Driving Terms

`simplestoragering` calculates the 3rd- and 4th-order RDTs and their fluctuations.

The perturbation from the $i$-th sextupole in the normal form is denoted as $\hat{V}_i$. 
The third-order generator for the ring Hamiltonian is $h_{3} = \sum_{i=1}^N \hat{V}_i$, where $N$ is the number of sextupoles of the ring.
The fourth-order generator is produced by the cross terms:
$$h_4 = \sum_{j>i=1}^{N}\left[\hat{V}_i, \hat{V}_j\right] = \sum_{j=2}^{N}\left[\sum_{i=1}^{j-1}\hat{V}_i, \hat{V}_j\right]$$

By recording the third-order RDT fluctuations, we can directly obtain the values of $\sum_{i=1}^{j-1}\hat{V}_i$,
reducing the calculation of the 4th-order RDTs to a **single loop** and **greatly improving the computational efficiency**.

The number of iterations is reduced from $N(N+1)/2$ to $N$.

In [ ]:
ring.linear_optics()
rdts = ring.driving_terms(verbose=True)

# The ADTS terms are driven by h22000, h11110, and h00220.
print('[dnux/dJx, dnuy/dJx, dnuy/dJy] = ', rdts.adts())
# print(f"\n{'Calculate ADTSs with RDTs':>50}:\n{'dQxx':>20}: {-4 * rdts['h2200'] / np.pi:.0f}, dQxy: {-2 * rdts['h1111'] / np.pi:.0f}, dQyy: {-4 * rdts['h0022'] / np.pi:.0f}")

## Longitudinal variations of RDTs

* Minimizing the RDT fluctuations can more effectively enlarge the DA area than minimizing the commonly used one-turn RDTs.
* By reducing the variation of lower-order RDTs along the longitudinal position of a storage ring, 
the cross terms contributing to higher-order nonlinear terms are reduced, 
which in turn suppresses higher-order resonances and controls ADTSs.

References:
> [1] IPAC'23 WEPL078  
> [2] [Minimizing the fluctuation of resonance driving terms in dynamic aperture optimization](https://link.aps.org/doi/10.1103/PhysRevAccelBeams.26.084001)


In [ ]:
# Build-up fluctuation: calculate nonlinear maps that share the same starting position but end at different positions, showing the build-up and cancellation of RDTs along the ring.
data = ssr.plot_RDTs_along_ring(ring, 'h')
# Natural fluctuation: RDTs are calculated in the normal form for the nonlinear case, with a varying starting position.
data = ssr.plot_RDTs_along_ring(ring, 'f')

In [ ]:
rdts_fluct = rdts.buildup_fluctuation(n_periods=14)  # RDT fluctuations in the complex plane.
fig = plt.figure(figsize=(10.5, 10))
plt.subplots_adjust(left=0.05, right=0.98, bottom=0.05, top=0.95, wspace=0.3)
for i, k in enumerate(['h2100', 'h3000', 'h1011', 'h1002', 'h1020',
                       'h3100', 'h4000', 'h2011', 'h1120', 'h2002', 'h2020', 'h0031', 'h0040']):
    plt.subplot(4, 4, i + 1)
    plt.scatter(np.real(rdts_fluct[k]), np.imag(rdts_fluct[k]), s=5)
    plt.text(0.99, 0.01, k, transform=plt.gca().transAxes, size=15, horizontalalignment="right")
plt.suptitle('RDT fluctuations in the complex plane')
plt.show()

Third-order RDT fluctuations have a simple structure.

Let $N$ be the number of sextupoles in a cell, and let $w$ be the index of the $w$-th sextupole in the cell,
the third-order RDT $h_{\vec{m}}$ at $(k N + w)$-th sextupole with k as a variable has two terms. 

For different sextupole index $w$, the $e^{i\vec{m}\cdot \vec{\mu}}$ terms of different $i$ are diffenrnt and the constant terms are the same. So there are concentric circles in the complex plane. 
When $w=N$, i.e. RDTs of multi-cell map, the third-order RDTs lie on a circle passing through the origin.

The fluctuation of fourth-order RDTs is more complex. see [[2]](https://link.aps.org/doi/10.1103/PhysRevAccelBeams.26.084001)
There is also constant term and $e^{i\vec{m}\cdot\vec{\mu}}$ terms for each $w$. And there are also $e^{i\vec{m}_1\cdot\vec{\mu}}$ terms and $e^{i\vec{m}_2\cdot\vec{\mu}}$ terms, where $\vec{m} = \vec{m}_1 + \vec{m}_2$.

In [ ]:
N = int(len(rdts_fluct['h2100']) / 14)
N_cell = 100
multi_cell_fluct = rdts.buildup_fluctuation(n_periods=N_cell)
fluct_comp = rdts.fluctuation_components()

fig = plt.figure(figsize=(10.5, 5))
plt.subplots_adjust(left=0.05, right=0.98, bottom=0.05, top=0.95, wspace=0.3)

for _, i in enumerate([0, 3, 9]):
    plt.subplot(2, 4, _ + 1)
    for k in range(N_cell):
        plt.scatter(np.real(multi_cell_fluct['h2100'][int(k*N+i)]), np.imag(multi_cell_fluct['h2100'][int(k*N+i)]), c='C0', s=5)
    n_cell = 0
    r0 = complex(0, 0)
    for ratio, radius in fluct_comp['h2100']:
        r1 = radius[i] * ratio ** n_cell
        plt.arrow(r0.real, r0.imag, r1.real, r1.imag, length_includes_head=True, width=abs(r1) / 20)
        r0 = r0 + r1
    plt.text(0.99, 0.01, f'h21000\n$k$ N + {i+1}', transform=plt.gca().transAxes, size=15, horizontalalignment="right")
    plt.subplot(2, 4, _ + 5)
    for k in range(N_cell):
        plt.scatter(np.real(multi_cell_fluct['h3100'][int(k*N+i)]), np.imag(multi_cell_fluct['h3100'][int(k*N+i)]), c='C0', s=5)
    n_cell = 9
    r0 = complex(0, 0)
    for ratio, radius in fluct_comp['h3100']:
        r1 = radius[i] * ratio ** n_cell
        plt.arrow(r0.real, r0.imag, r1.real, r1.imag, length_includes_head=True, width=abs(r1) / 20)
        r0 = r0 + r1
    plt.text(0.99, 0.01, f'h31000\n$k$ N + {i+1}', transform=plt.gca().transAxes, size=15, horizontalalignment="right")

    plt.subplot(2, 4, 4)
for k in range(N_cell):
    plt.scatter(np.real(multi_cell_fluct['h2100'][int(k*N - 1)]), np.imag(multi_cell_fluct['h2100'][int(k*N-1)]), c='C0', s=5)
n_cell = 10
r0 = complex(0, 0)
for ratio, radius in fluct_comp['h2100']:
    r1 = radius[-1] * ratio ** n_cell
    plt.arrow(r0.real, r0.imag, r1.real, r1.imag, length_includes_head=True, width=abs(r1) / 20)
    r0 = r0 + r1
plt.text(0.99, 0.01, f'h21000\n$k$ N + N', transform=plt.gca().transAxes, size=15, horizontalalignment="right")

plt.subplot(2, 4, 8)
for k in range(N_cell):
    plt.scatter(np.real(multi_cell_fluct['h3100'][int(k*N - 1)]), np.imag(multi_cell_fluct['h3100'][int(k*N-1)]), c='C0', s=5)
n_cell = 1
r0 = complex(0, 0)
for ratio, radius in fluct_comp['h3100']:
    r1 = radius[-1] * ratio ** n_cell
    plt.arrow(r0.real, r0.imag, r1.real, r1.imag, length_includes_head=True, width=abs(r1) / 20)
    r0 = r0 + r1
plt.text(0.99, 0.01, f'h31000\n$k$ N + N', transform=plt.gca().transAxes, size=15, horizontalalignment="right")
plt.suptitle('RDT fluctuations')
plt.show()